# Vehicle Damage Classification Using Deep Learning

**Course:** COMP-671 Deep Learning  
**Project:** Seven-class vehicle damage image classification

This is an organized **submission copy** of the original Colab notebook. The original notebook and its training outputs are preserved separately. Results below are **recorded experimental outcomes**, not newly computed by this notebook. The notebook defaults to loading saved artifacts and does **not retrain** models.

**Important:** A comparison between the two ResNet50 runs is observational: some training configurations differed, so the 0.69 percentage-point gap cannot be attributed solely to augmentation.

## 1. Experiment overview

- Labeled images: **963** (unlabeled `unknown` entries excluded)
- Stratified split: **674 train / 144 validation / 145 test**
- Seven classes: `bumper_dent`, `bumper_scratch`, `door_dent`, `door_scratch`, `glass_shatter`, `head_lamp`, `tail_lamp`
- Models: a small baseline CNN, frozen ImageNet ResNet50 without augmentation, and frozen ImageNet ResNet50 with augmentation
- Evaluation: held-out test accuracy, class-wise precision/recall/F1, confusion matrix, and error inspection

**Dataset source and reuse permissions:** Please add the original dataset URL and license before publishing the dataset itself.

## 2. Configuration and dependencies

In [ ]:
from pathlib import Path
import os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display, Image

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
CLASS_NAMES = ['bumper_dent', 'bumper_scratch', 'door_dent', 'door_scratch', 'glass_shatter', 'head_lamp', 'tail_lamp']
PROJECT_DIR = Path('/content/drive/MyDrive/Vehicle Damage Project')
RESULTS_DIR = PROJECT_DIR / 'results'
MODELS_DIR = PROJECT_DIR / 'models'
DATA_DIR = Path('/content/vehicle_damage_data')

# In Google Colab, uncomment the following two lines to access saved artifacts:
# from google.colab import drive
# drive.mount('/content/drive')

np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Project directory:', PROJECT_DIR)
print('Drive accessible:', PROJECT_DIR.exists())

## 3. Dataset restoration and preprocessing

The dataset ZIP is expected at `Vehicle Damage Project/archive.zip` on Google Drive. The original dataset remains in Google Drive; it is **not** included in the GitHub submission. Image pixels are resized to 224 × 224 and remain in the 0–255 float range. The baseline CNN rescales to [0, 1] internally, while ResNet50 uses its own `preprocess_input`.

The three saved CSV split files are preferred over generating new splits, preserving the original held-out evaluation.

In [ ]:
zip_path = PROJECT_DIR / 'archive.zip'
if not (DATA_DIR / 'image').exists():
    if zip_path.is_file():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(DATA_DIR)
        print('Dataset restored from Google Drive')
    else:
        print('Dataset not found. Mount Drive and place archive.zip in the project folder.')

splits_ready = all((PROJECT_DIR / f'{name}_split.csv').is_file() for name in ('train','val','test'))
if splits_ready:
    train_df = pd.read_csv(PROJECT_DIR / 'train_split.csv')
    val_df = pd.read_csv(PROJECT_DIR / 'val_split.csv')
    test_df = pd.read_csv(PROJECT_DIR / 'test_split.csv')
    for frame in (train_df, val_df, test_df):
        frame['full_path'] = frame['image'].astype(str).map(lambda x: str(DATA_DIR / x))
        frame['label'] = frame['classes'].map({name:i for i,name in enumerate(CLASS_NAMES)}).astype(int)
    assert len(train_df)==674 and len(val_df)==144 and len(test_df)==145
    assert set(train_df.image).isdisjoint(set(val_df.image))
    assert set(train_df.image).isdisjoint(set(test_df.image))
    assert set(val_df.image).isdisjoint(set(test_df.image))
    print('Splits restored:', len(train_df),len(val_df),len(test_df))
else:
    print('Saved split CSVs not found; copy them from the original project folder.')

In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    return tf.cast(image, tf.float32), tf.cast(label, tf.int32)

def make_dataset(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((frame['full_path'].tolist(), frame['label'].tolist()))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

if splits_ready and (DATA_DIR / 'image').exists():
    train_ds = make_dataset(train_df, shuffle=True)
    val_ds = make_dataset(val_df)
    test_ds = make_dataset(test_df)
    print('Datasets ready')
else:
    print('Dataset unavailable: data-dependent evaluation cells can be run after restoration.')

## 4. Model architectures and training

The following cells document the model configurations. **They do not train or overwrite saved weights.** Training was performed in the original notebook, with the best model selected by validation accuracy and saved to Google Drive.

The baseline CNN has 110,663 trainable parameters. ResNet50 uses a frozen ImageNet pretrained feature extractor, global average pooling, a 128-unit dense layer, dropout, and a seven-way softmax classifier.

In [ ]:
from tensorflow.keras import layers

def build_baseline_cnn():
    model = tf.keras.Sequential([
        layers.Input(shape=(224,224,3)), layers.Rescaling(1./255),
        layers.Conv2D(32,3,activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(64,3,activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(128,3,activation='relu'), layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(), layers.Dense(128,activation='relu'),
        layers.Dropout(0.3), layers.Dense(7,activation='softmax')
    ], name='Baseline_CNN')
    return model

baseline_architecture = build_baseline_cnn()
baseline_architecture.summary()

In [ ]:
def build_resnet50(use_augmentation=False, learning_rate=0.001):
    backbone = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(224,224,3))
    x = inputs
    if use_augmentation:
        x = tf.keras.Sequential([
            layers.RandomFlip('horizontal'), layers.RandomRotation(0.08),
            layers.RandomZoom(0.10)
        ], name='data_augmentation')(x)
    x = tf.keras.applications.resnet50.preprocess_input(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(7, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Rebuild only if you need to inspect the architecture or restore .weights.h5.
# Do not call model.fit() unless intentionally repeating the experiment.

### 4.1 Saved model artifacts

- `models/baseline_cnn_best.keras`
- `resnet_no_augmentation.weights.h5` (weights only; requires rebuilding the matching architecture)
- `models/resnet50_aug_best.keras`

**Reproducibility note:** The non-augmented ResNet50 was run more than once with differing configurations. The saved weights and reported historical metrics may not correspond to a single fully controlled experiment. Refer to the original notebook for the full training sequence.

## 5. Recorded test results

In [ ]:
# Historical test accuracies from the completed project, NOT newly evaluated here.
recorded = pd.DataFrame({
    'Model':['Baseline CNN','ResNet50 Without Augmentation','ResNet50 With Augmentation'],
    'Test Accuracy (%)':[42.068964,77.93,78.62]
})
display(recorded)
ax = recorded.plot.bar(x='Model', y='Test Accuracy (%)', legend=False, figsize=(9,5), rot=10)
ax.set_ylim(0,100); ax.set_ylabel('Test Accuracy (%)'); ax.set_title('Recorded model comparison')
plt.tight_layout(); plt.show()
print('ResNet50 augmentation gap: 0.69 percentage points (not a controlled causal comparison).')

### 5.1 Augmented ResNet50 classification metrics

The recorded augmented ResNet50 test evaluation returned **78.62% accuracy** and **0.5573 loss** over **145 images**. The macro precision, recall and F1 were approximately **0.81, 0.78 and 0.78** respectively. The classification report and confusion matrix below are from the original run.

In [ ]:
# This reads the actual CSV if present in Drive; alternatively use results/ next to this notebook in GitHub.
local_results = Path('results')
report_file = RESULTS_DIR / 'resnet50_aug_classification_report.csv'
if not report_file.exists():
    report_file = local_results / 'resnet50_aug_classification_report.csv'
if report_file.exists():
    display(pd.read_csv(report_file).rename(columns={'Unnamed: 0':'Class'}))
else:
    print('Classification report CSV not found; see results/ in GitHub package.')

In [ ]:
def show_saved_figure(name):
    path = RESULTS_DIR / name
    if not path.exists(): path = Path('results') / name
    if path.exists(): display(Image(filename=str(path)))
    else: print('Figure unavailable:', name)

show_saved_figure('resnet50_aug_confusion_matrix.png')
show_saved_figure('misclassified_images.png')

### 5.2 Failure analysis

From the recorded confusion matrix, **31 of 145 images** were misclassified. In particular, **12 door-dent** images were predicted as door scratches, and **5 bumper-scratch** images were predicted as door scratches. All **21 glass-shatter** test images were correctly classified. Reflections, lighting and similar surface patterns are plausible contributing factors, but these were not experimentally isolated.

## 6. Optional: independently re-evaluate the saved models

This section is safe to run when Google Drive and `test_ds` are available. It **loads saved models**, without retraining. It does not overwrite recorded results.

In [ ]:
if 'test_ds' in globals() and (MODELS_DIR/'baseline_cnn_best.keras').exists():
    model = tf.keras.models.load_model(MODELS_DIR/'baseline_cnn_best.keras', compile=False)
    model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    loss, accuracy = model.evaluate(test_ds,verbose=0)
    print(f'Baseline CNN test accuracy: {accuracy:.4%}; loss: {loss:.4f}')
else:
    print('Baseline evaluation skipped: restore test dataset and saved model.')

if 'test_ds' in globals() and (MODELS_DIR/'resnet50_aug_best.keras').exists():
    model = tf.keras.models.load_model(MODELS_DIR/'resnet50_aug_best.keras', compile=False)
    model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    loss, accuracy = model.evaluate(test_ds,verbose=0)
    print(f'Augmented ResNet50 test accuracy: {accuracy:.4%}; loss: {loss:.4f}')
else:
    print('Augmented ResNet50 evaluation skipped: restore test dataset and saved model.')

## 7. Limitations and conclusion

The pretrained ResNet50 models outperformed the small baseline CNN on the recorded held-out split. The augmented ResNet50 attained the highest observed test accuracy (78.62%). However, the test set is small, some damage classes overlap visually, and the non-augmented and augmented ResNet50 training configurations were not fully controlled. Larger, more diverse datasets and controlled repeated experiments would help assess generalization and the effect of augmentation.

## 8. References and provenance

- He, K., Zhang, X., Ren, S., & Sun, J. (2016). *Deep Residual Learning for Image Recognition*. CVPR.
- TensorFlow/Keras documentation for `ResNet50`, image augmentation, and model checkpoints.
- **Dataset:** Original `archive.zip` with `data.csv` and `image/` directory. **Add its verified original source URL/license before submission.**

**Provenance:** This notebook is a cleaned, reader-oriented version of `Vehicle_Damage_Classification(1).ipynb`. The original experiment notebook should be retained for full training logs and procedural history. The saved CSV/PNG figures in `results/` come from the uploaded project materials.